# Statement analytics

**Purpose:** Use `finstack.statements_analytics` to stress-test, compare, explain, and score forecasts on a statement model built with `ModelBuilder`.

**Prerequisites:** Work through **notebook 09** in this track (statement modeling) so you are comfortable building periods, value nodes, compute nodes, and evaluating with `Evaluator`.


## Analytics on statement models

A financial model is a directed graph of periods and nodes. Once you can evaluate it, analytics answers adjacent questions: *What if drivers move?* *How does a new plan differ from the old one?* *What revenue clears a target EBITDA?* *Which scenarios matter?* *Where did this number come from?* *Was the forecast any good?*

The Python bindings expose these workflows as small functions that take JSON (or simple numeric vectors) and return JSON or structured Python values—ideal for notebooks, dashboards, and automated reports.


## Shared baseline: a compact P&L

We build one demo model and serialize it with `spec.to_json()`. Evaluation produces `eval_json` via `result.to_json()`. Later calls reuse these strings so each analytics API stays a pure function of model + configuration.


In [ ]:
import json

from finstack.statements import Evaluator, ModelBuilder

b = ModelBuilder("analytics-demo")
b.periods("2025Q1..Q4", "2025Q2")
b.value("revenue", [("2025Q1", 100.0), ("2025Q2", 110.0), ("2025Q3", 115.0), ("2025Q4", 120.0)])
b.value("cogs", [("2025Q1", 60.0), ("2025Q2", 65.0), ("2025Q3", 68.0), ("2025Q4", 72.0)])
b.compute("gross_profit", "revenue - cogs")
b.value("opex", [("2025Q1", 20.0), ("2025Q2", 22.0), ("2025Q3", 23.0), ("2025Q4", 24.0)])
b.compute("ebitda", "gross_profit - opex")
spec = b.build()
model_json = spec.to_json()
result = Evaluator().evaluate(spec)
eval_json = result.to_json()

print("Model id:", json.loads(model_json)["id"])
print("EBITDA 2025Q1 (evaluated):", json.loads(eval_json)["nodes"]["ebitda"]["2025Q1"])


## Sensitivity and tornado views

`run_sensitivity` perturbs explicit driver nodes and records how target metrics respond. `generate_tornado_entries` narrows that JSON to a ranked list for one metric (and optional period)—useful for plotting.


In [ ]:
from finstack.statements_analytics import generate_tornado_entries, run_sensitivity

sensitivity_config = json.dumps({
    "mode": "Diagonal",
    "parameters": [{
        "node_id": "revenue",
        "period_id": "2025Q1",
        "base_value": 100.0,
        "perturbations": [-20.0, -10.0, -5.0, 0.0, 5.0, 10.0, 20.0],
    }],
    "target_metrics": ["gross_profit", "ebitda"],
})
sens_result = run_sensitivity(model_json, sensitivity_config)
print(sens_result)

tornado = generate_tornado_entries(sens_result, "ebitda", "2025Q1")
print(tornado)


## Variance between two evaluated models

`run_variance` compares two **evaluation** JSON payloads (baseline vs comparison) for selected metrics and periods. Build a second `ModelBuilder`, evaluate it, and pass both `to_json()` outputs plus a `VarianceConfig`.


In [ ]:
from finstack.statements_analytics import run_variance

b2 = ModelBuilder("comparison")
b2.periods("2025Q1..Q4", "2025Q2")
b2.value("revenue", [("2025Q1", 105.0), ("2025Q2", 115.0), ("2025Q3", 120.0), ("2025Q4", 125.0)])
b2.value("cogs", [("2025Q1", 62.0), ("2025Q2", 67.0), ("2025Q3", 70.0), ("2025Q4", 74.0)])
b2.compute("gross_profit", "revenue - cogs")
b2.value("opex", [("2025Q1", 21.0), ("2025Q2", 23.0), ("2025Q3", 24.0), ("2025Q4", 25.0)])
b2.compute("ebitda", "gross_profit - opex")
comp_result = Evaluator().evaluate(b2.build())

variance_config = json.dumps({
    "baseline_label": "base",
    "comparison_label": "upside",
    "metrics": ["gross_profit", "ebitda"],
    "periods": ["2025Q1", "2025Q2"],
})
variance = run_variance(eval_json, comp_result.to_json(), variance_config)
print(variance)


## Goal seek

`goal_seek` searches for a driver value so a target node hits a desired level in a given period. With `update_model=False`, the returned model JSON is still updated in the second tuple element per the binding contract—print the solved driver and a short preview of the updated model.


In [ ]:
from finstack.statements_analytics import goal_seek

gs_result = goal_seek(
    model_json,
    target_node="ebitda",
    target_period="2025Q1",
    target_value=25.0,
    driver_node="revenue",
    driver_period="2025Q1",
    update_model=False,
)
solved_revenue, updated_model_json = gs_result
print("Solved revenue (2025Q1):", solved_revenue)
print("Updated model JSON (first 400 chars):", updated_model_json[:400])


## Scenario set evaluation

`evaluate_scenario_set` runs every named scenario against the same base model. Each scenario supplies `overrides` as **node id → scalar**; those scalars are applied across **forecast** periods in the model (see library semantics). Below, upside and downside shift `revenue` everywhere it acts as a forecast driver.


In [ ]:
from finstack.statements_analytics import evaluate_scenario_set

scenario_set = json.dumps({
    "scenarios": {
        "upside": {"overrides": {"revenue": 130.0}},
        "downside": {"overrides": {"revenue": 100.0}},
    }
})
scenario_results = evaluate_scenario_set(model_json, scenario_set)
print(scenario_results)


## Dependencies and formula explanation

`trace_dependencies` returns an ASCII tree of upstream nodes. `direct_dependencies` lists immediate inputs. `explain_formula` returns a **Python dict** (JSON-serializable) with the resolved breakdown for one node and period—use `json.dumps` for a stable string view.


In [ ]:
from finstack.statements_analytics import direct_dependencies, explain_formula, trace_dependencies

deps = trace_dependencies(model_json, "ebitda")
print("Full dependency tree:")
print(deps)

direct = direct_dependencies(model_json, "ebitda")
print("Direct deps:", direct)

explanation = explain_formula(model_json, eval_json, "ebitda", "2025Q1")
print("Formula explanation (JSON):", json.dumps(explanation, indent=2))


## Backtest forecast accuracy

`backtest_forecast` expects parallel sequences of actuals and forecasts (same length) and returns MAE, MAPE, RMSE, and count. Here we align two quarters of revenue actuals vs the baseline forecast.


In [ ]:
from finstack.statements_analytics import backtest_forecast

actual = [102.0, 108.0]
forecast = [100.0, 110.0]
bt = backtest_forecast(actual, forecast)
print(json.dumps(dict(bt)))


## Mini-example: full analytics pass on the demo P&L

The cell below is self-contained: it rebuilds the baseline and comparison models, then runs sensitivity, variance, scenarios, goal seek, dependency tracing, formula explanation, and a tiny revenue backtest—mirroring how you might script a quarterly analytics pack.


In [ ]:
import json

from finstack.statements import Evaluator, ModelBuilder
from finstack.statements_analytics import (
    backtest_forecast,
    direct_dependencies,
    evaluate_scenario_set,
    explain_formula,
    generate_tornado_entries,
    goal_seek,
    run_sensitivity,
    run_variance,
    trace_dependencies,
)

# Baseline
b = ModelBuilder("analytics-demo")
b.periods("2025Q1..Q4", "2025Q2")
b.value("revenue", [("2025Q1", 100.0), ("2025Q2", 110.0), ("2025Q3", 115.0), ("2025Q4", 120.0)])
b.value("cogs", [("2025Q1", 60.0), ("2025Q2", 65.0), ("2025Q3", 68.0), ("2025Q4", 72.0)])
b.compute("gross_profit", "revenue - cogs")
b.value("opex", [("2025Q1", 20.0), ("2025Q2", 22.0), ("2025Q3", 23.0), ("2025Q4", 24.0)])
b.compute("ebitda", "gross_profit - opex")
spec = b.build()
model_json = spec.to_json()
base_eval = Evaluator().evaluate(spec).to_json()

# Comparison (upside plan)
b2 = ModelBuilder("comparison")
b2.periods("2025Q1..Q4", "2025Q2")
b2.value("revenue", [("2025Q1", 105.0), ("2025Q2", 115.0), ("2025Q3", 120.0), ("2025Q4", 125.0)])
b2.value("cogs", [("2025Q1", 62.0), ("2025Q2", 67.0), ("2025Q3", 70.0), ("2025Q4", 74.0)])
b2.compute("gross_profit", "revenue - cogs")
b2.value("opex", [("2025Q1", 21.0), ("2025Q2", 23.0), ("2025Q3", 24.0), ("2025Q4", 25.0)])
b2.compute("ebitda", "gross_profit - opex")
cmp_eval = Evaluator().evaluate(b2.build()).to_json()

print("=== Sensitivity (EBITDA tornado, 2025Q1) ===")
sens_cfg = json.dumps({
    "mode": "Diagonal",
    "parameters": [{
        "node_id": "revenue",
        "period_id": "2025Q1",
        "base_value": 100.0,
        "perturbations": [-20.0, -10.0, -5.0, 0.0, 5.0, 10.0, 20.0],
    }],
    "target_metrics": ["gross_profit", "ebitda"],
})
sens = run_sensitivity(model_json, sens_cfg)
print(generate_tornado_entries(sens, "ebitda", "2025Q1"))

print("=== Variance (Q1–Q2, gross_profit & ebitda) ===")
var_cfg = json.dumps({
    "baseline_label": "base",
    "comparison_label": "upside",
    "metrics": ["gross_profit", "ebitda"],
    "periods": ["2025Q1", "2025Q2"],
})
print(run_variance(base_eval, cmp_eval, var_cfg))

print("=== Scenarios (revenue shocks) ===")
scen = json.dumps({
    "scenarios": {
        "upside": {"overrides": {"revenue": 130.0}},
        "downside": {"overrides": {"revenue": 100.0}},
    }
})
print(evaluate_scenario_set(model_json, scen))

print("=== Goal seek EBITDA 2025Q1 -> 25 ===")
solved, _mj = goal_seek(
    model_json,
    target_node="ebitda",
    target_period="2025Q1",
    target_value=25.0,
    driver_node="revenue",
    driver_period="2025Q1",
    update_model=False,
)
print("Solved revenue:", solved)

print("=== Dependencies for ebitda ===")
print(trace_dependencies(model_json, "ebitda"))
print("Direct:", direct_dependencies(model_json, "ebitda"))
print("Explain:", json.dumps(explain_formula(model_json, base_eval, "ebitda", "2025Q1")))

print("=== Revenue forecast backtest (2 quarters) ===")
print(json.dumps(dict(backtest_forecast([102.0, 108.0], [100.0, 110.0]))))


## Takeaways

- **Sensitivity** (`run_sensitivity`, `generate_tornado_entries`) quantifies how driver shocks flow into key metrics before you commit to a plan.
- **Variance** (`run_variance`) formalizes baseline vs alternate evaluation JSON—ideal for budget vs actual or plan versions.
- **Goal seek** finds the driver level that hits a target metric in a period, preserving the rest of the model structure.
- **Scenarios** (`evaluate_scenario_set`) batch-evaluate named override sets; overrides are node-level scalars broadcast across forecast periods.
- **Introspection** (`trace_dependencies`, `direct_dependencies`, `explain_formula`) supports audit trails and IC-ready narrative around a number.
- **Backtesting** (`backtest_forecast`) scores vector forecasts with standard error metrics.

**Next:** Keep building richer statement models in notebook 09’s spirit, then wire these analytics into your own reporting or scenario libraries.
